In [19]:
import sys
import pandas as pd
import numpy as np
import matplotlib
import sklearn

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.11.17
pandas: 3.0.6
NumPy: 2.4.6
Matplotlib: 3.11.0
scikit-learn: 1.9.1


In [20]:
import sys
import pandas as pd
import numpy as np
import matplotlib
import sklearn

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.11.17
pandas: 3.0.6
NumPy: 2.4.6
Matplotlib: 3.11.0
scikit-learn: 1.9.1


In [21]:
import sys
print(sys.executable)

C:\Users\User1\anaconda3\envs\ct4101project\python.exe


In [22]:
import pandas as pd

df = pd.read_csv("../data/netflix_customer_churn.csv")

df.head()

,customer_id,age,gender,subscription_type,watch_hours,last_login_days,region,device,monthly_fee,churned,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre
0,a9b75100-82a8-427a-a208-72f24052884a,51,Other,Basic,14.73,29,Africa,TV,8.99,1,Gift Card,1,0.49,Action
1,49a5dfd9-7e69-4022-a6ad-0a1b9767fb5b,47,Other,Standard,0.70,19,Europe,Mobile,13.99,1,Gift Card,5,0.03,Sci-Fi
2,4d71f6ce-fca9-4ff7-8afa-197ac24de14b,27,Female,Standard,16.32,10,Asia,TV,13.99,0,Crypto,2,1.48,Drama
3,d3c72c38-631b-4f9e-8a0e-de103cad1a7d,53,Other,Premium,4.51,12,Oceania,TV,17.99,1,Crypto,2,0.35,Horror
4,4e265c34-103a-4dbb-9553-76c9aa47e946,56,Other,Standard,1.89,13,Africa,Mobile,13.99,1,Crypto,2,0.13,Action


Data Audit, Dataset Shape and Variable Types

In [23]:
print("Number of records:", df.shape[0])
print("Number of variables:", df.shape[1])

Number of records: 5000
Number of variables: 14


In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_id             5000 non-null   str    
 1   age                     5000 non-null   int64  
 2   gender                  5000 non-null   str    
 3   subscription_type       5000 non-null   str    
 4   watch_hours             5000 non-null   float64
 5   last_login_days         5000 non-null   int64  
 6   region                  5000 non-null   str    
 7   device                  5000 non-null   str    
 8   monthly_fee             5000 non-null   float64
 9   churned                 5000 non-null   int64  
 10  payment_method          5000 non-null   str    
 11  number_of_profiles      5000 non-null   int64  
 12  avg_watch_time_per_day  5000 non-null   float64
 13  favorite_genre          5000 non-null   str    
dtypes: float64(3), int64(4), str(7)
memory usage: 547.0

In [25]:
# check missing values
print(df.isnull().sum())

# Total missing values
print("\nTotal missing values:", df.isnull().sum().sum())

customer_id               0
age                       0
gender                    0
subscription_type         0
watch_hours               0
last_login_days           0
region                    0
device                    0
monthly_fee               0
churned                   0
payment_method            0
number_of_profiles        0
avg_watch_time_per_day    0
favorite_genre            0
dtype: int64

Total missing values: 0


This missing value check shows no missing values across the 14 variables and 5,000 records, so no missing-value handling or imputation needs to be done here before modelling.

Duplicate records

This section checks if the dataset has any duplicate rows or repeated customers

In [26]:
# Check for completely identical rows
print("Duplicate rows:", df.duplicated().sum())

# Check for repated customer IDs
print("Duplicate customer IDs:", df["customer_id"].duplicated().sum())

Duplicate rows: 0
Duplicate customer IDs: 0


These duplicate checks show there are no duplicated rows or customer IDs in the dataset, so no duplicate records need to be removed during preprocessing.

Target Distribution

This section checks the distribution of the target variable (churned) to see if the classes are balanced

In [27]:
#Count the number of customers in each class
print("Churn counts:")
print(df["churned"].value_counts().sort_index())

#calculate the percentage of customers in each class
print("\nChurn Percentage:")
print(df["churned"].value_counts(normalize=True).sort_index() * 100)

Churn counts:
churned
0    2485
1    2515
Name: count, dtype: int64

Churn Percentage:
churned
0    49.7
1    50.3
Name: proportion, dtype: float64


The results show 2,485 customers (49.7%) did not churn and 2,515 customers (50.3%) churned. This shows this dataset is well balanced, so oversampling or undersampling should not be necessary

Outlier Analysis

In [28]:
df.describe()

,age,watch_hours,last_login_days,monthly_fee,churned,number_of_profiles,avg_watch_time_per_day
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,43.847400,11.649450,30.089800,13.683400,0.503000,3.024400,0.874800
std,15.501128,12.014654,17.536078,3.692062,0.500041,1.415841,2.619824
min,18.000000,0.010000,0.000000,8.990000,0.000000,1.000000,0.000000
25%,30.000000,3.337500,15.000000,8.990000,0.000000,2.000000,0.110000
50%,44.000000,8.000000,30.000000,13.990000,1.000000,3.000000,0.290000
75%,58.000000,16.030000,45.000000,17.990000,1.000000,4.000000,0.720000
max,70.000000,110.400000,60.000000,17.990000,1.000000,5.000000,98.420000


Identifying Potential Outliers Using Interquartile Range

In [29]:
columns = ["watch_hours", "avg_watch_time_per_day"]

for column in columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    print(column)
    print("Lower bound:", round(lower_bound, 2))
    print("Upper bound:", round(upper_bound, 2))
    print("Number of potential outliers:", len(outliers))
    print()

watch_hours
Lower bound: -15.7
Upper bound: 35.07
Number of potential outliers: 238

avg_watch_time_per_day
Lower bound: -0.8
Upper bound: 1.64
Number of potential outliers: 549



In [30]:
X = df.drop(columns=["customer_id", "churned"])
y = df["churned"]

print("Input features:", X.shape)
print("Target variable:", y.shape)

Input features: (5000, 12)
Target variable: (5000,)


I split the dataset into 70% Training (3,500 records), 20% Validation (1,000) and 10% Testing (500 records). The training set is used to train the models, the validation set is used to compare and evaluate the models and the test set is kept for final evaluation. I used stratified splitting to keep similar portions of churned and retained customers across each of the 3 sets. The fixed random state is set to 42 so that the same training, validation and test sets are generated every time this code is run.

In [31]:
from sklearn.model_selection import train_test_split

# Split into 70% training and 30% remaining
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Divide remaining data into 20% validation and 10% testing
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=1/3,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (3500, 12)
Validation: (1000, 12)
Testing: (500, 12)


Preprocessing Pipeline

The customer_id variable is excluded because it is an identifier and churned was separated as it is the target variable. The 12 input features left have 6 numerical and 6 categorical variables. I used StandardScaler to standardise the numerical features and encoded the categorical features using OneHotEncoder. This preprocessing pipeline is only fitted on the training data to prevent data leakage.

In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify numerical and categorical features
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["str", "object", "category"]
).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

# Create the preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

Numerical features: ['age', 'watch_hours', 'last_login_days', 'monthly_fee', 'number_of_profiles', 'avg_watch_time_per_day']
Categorical features: ['gender', 'subscription_type', 'region', 'device', 'payment_method', 'favorite_genre']


Baseline Model- Dummy Classifier

In [33]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline

# Create the baseline model with preprocessing
baseline_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent"))
])

# Train the baseline model using only the training data
baseline_model.fit(X_train, y_train)

print("Baseline model trained successfully!")

Baseline model trained successfully!


Baseline Model Evaluation

In [34]:
# Predict churn using the validation dataset
y_val_pred = baseline_model.predict(X_val)

# Generate predicted probabilities for churn (class 1)
y_val_prob = baseline_model.predict_proba(X_val)[:, 1]

print("Predictions generated successfully!")

Predictions generated successfully!


In [35]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Calculate evaluation metrics
accuracy = accuracy_score(y_val, y_val_pred)
precision = precision_score(y_val, y_val_pred, zero_division=0)
recall = recall_score(y_val, y_val_pred, zero_division=0)
f1 = f1_score(y_val, y_val_pred, zero_division=0)
roc_auc = roc_auc_score(y_val, y_val_prob)

# Display results
print("Baseline Model Evaluation Results")
print("---------------------------------")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

Baseline Model Evaluation Results
---------------------------------
Accuracy:  0.5030
Precision: 0.5030
Recall:    1.0000
F1-score:  0.6693
ROC-AUC:   0.5000
